In [5]:
import json
import base64
from openai import OpenAI

In [6]:
client = OpenAI(
    api_key="os.environ.get("OPENAI_API_KEY")"
)

In [7]:
SYSTEM_PROMPT = """\
You are a surgical answer-matching evaluator. Your job is to compare two answers and decide if they express the SAME MEANING, regardless of wording.

CRITICAL RULES:
1. Numbers match if they express the same quantity:
   - "2" matches "two" matches "Two surgical instruments"
   - "silver/grey" matches "silver" or "grey"
   - Count the core information, not the wording

2. Match if answers convey IDENTICAL core meaning:
   - "right" matches "right side" or "right direction"
   - "bipolar" matches "the bipolar" or "the bipolar instrument"
   - Extra details are OK as long as core fact is the same

3. NO_MATCH only if:
   - Different numbers: "2" vs "3"
   - Contradictory facts: "left" vs "right"
   - Wrong entities: talking about different tools/structures
   - Completely different answers

4. Extract the CORE ANSWER first:
   - Answer 1: "2" → Core: quantity is 2
   - Answer 2: "Two surgical instruments are visible" → Core: quantity is 2
   - These are MATCH

EXAMPLES:
Answer 1: "silver/grey" | Answer 2: "The bipolar is blue" → NO_MATCH (colors don't match)
Answer 1: "right" | Answer 2: "The tip is directed toward the right side" → MATCH (both say right)
Answer 1: "2" | Answer 2: "Two surgical instruments are visible" → MATCH (both say quantity 2)
Answer 1: "I don't know" | Answer 2: "Unable to determine from image" → MATCH (both express inability)

Return ONLY:
MATCH
or
NO_MATCH
"""

In [8]:
def _parse_json(raw: str) -> dict:
    raw = raw.strip()
    if raw.startswith("```"):
        raw = raw.strip("`")
        if raw.lower().startswith("json"):
            raw = raw[4:]
    return json.loads(raw.strip())

In [9]:
def _encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()

In [10]:
def compare_answers(answer1: str, answer2: str) -> str:
    """Compare two answers and return whether they match in meaning.

    Returns: "MATCH" or "NO_MATCH"
    """

    user_text = (
        f"Answer 1: {answer1}\n"
        f"Answer 2: {answer2}"
    )

    try:
        response = client.responses.create(
            model="gpt-5.4-mini",
            instructions=SYSTEM_PROMPT,
            input=user_text,
        )

        result = response.output_text.strip()

        # Extract MATCH or NO_MATCH from response
        if "MATCH" in result.upper():
            if "NO_MATCH" in result.upper():
                no_match_idx = result.upper().find("NO_MATCH")
                match_idx = result.upper().find("MATCH")
                if no_match_idx < match_idx:
                    return "NO_MATCH"
            return "MATCH"
        else:
            return "NO_MATCH"

    except Exception as e:
        print(f"Error comparing answers: {e}")
        return None

In [11]:
# Load the JSON (change 4B to 7B or 14B as needed)
model_size = "4B"
json_file = f'/home/as5606/Datasets/Cholec_text_grounded_questions/hulumed_{model_size}_grounded_accuracy_comparison.json'

with open(json_file, 'r') as f:
    data = json.load(f)

In [13]:
from tqdm import tqdm
for item in tqdm(data):
    grounded_qa_list = item.get('grounded_qa_with_matches', [])
    if len(grounded_qa_list) > 3:
        fourth_qa = grounded_qa_list[3]
        original_answer = fourth_qa['original_answer']  # ← Change this
        hulu_answer = fourth_qa['hulu_med_answer']
        match_result = compare_answers(original_answer, hulu_answer)
        fourth_qa['match'] = match_result

# Save back
with open(json_file, 'w') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✓ Updated {json_file}")

100%|██████████| 1402/1402 [16:09<00:00,  1.45it/s] 


✓ Updated /home/as5606/Datasets/Cholec_text_grounded_questions/hulumed_4B_grounded_accuracy_comparison.json


In [14]:
# Load the JSON (change 4B to 7B or 14B as needed)
model_size = "7B"
json_file = f'/home/as5606/Datasets/Cholec_text_grounded_questions/hulumed_{model_size}_grounded_accuracy_comparison.json'

with open(json_file, 'r') as f:
    data = json.load(f)


from tqdm import tqdm
for item in tqdm(data):
    grounded_qa_list = item.get('grounded_qa_with_matches', [])
    if len(grounded_qa_list) > 3:
        fourth_qa = grounded_qa_list[3]
        original_answer = fourth_qa['original_answer']  # ← Change this
        hulu_answer = fourth_qa['hulu_med_answer']
        match_result = compare_answers(original_answer, hulu_answer)
        fourth_qa['match'] = match_result

# Save back
with open(json_file, 'w') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✓ Updated {json_file}")

100%|██████████| 1402/1402 [15:52<00:00,  1.47it/s]

✓ Updated /home/as5606/Datasets/Cholec_text_grounded_questions/hulumed_7B_grounded_accuracy_comparison.json


In [15]:
# Load the JSON (change 4B to 7B or 14B as needed)
model_size = "14B"
json_file = f'/home/as5606/Datasets/Cholec_text_grounded_questions/hulumed_{model_size}_grounded_accuracy_comparison.json'

with open(json_file, 'r') as f:
    data = json.load(f)


from tqdm import tqdm
for item in tqdm(data):
    grounded_qa_list = item.get('grounded_qa_with_matches', [])
    if len(grounded_qa_list) > 3:
        fourth_qa = grounded_qa_list[3]
        original_answer = fourth_qa['original_answer']  # ← Change this
        hulu_answer = fourth_qa['hulu_med_answer']
        match_result = compare_answers(original_answer, hulu_answer)
        fourth_qa['match'] = match_result

# Save back
with open(json_file, 'w') as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

print(f"✓ Updated {json_file}")

100%|██████████| 1402/1402 [16:58<00:00,  1.38it/s]

✓ Updated /home/as5606/Datasets/Cholec_text_grounded_questions/hulumed_14B_grounded_accuracy_comparison.json
